# Section 5: Creating the RAG Application

*Notes:* This notebook builds the retrieval-and-generation flow that uses the vector index and an LLM endpoint to answer user questions grounded in the video content.

The detailed background of this code is in this blog:



In [ ]:
from mlflow.deployments import get_deploy_client
from databricks.sdk import WorkspaceClient

VS_ENDPOINT   = "video_ai_vs_endpoint"
VS_INDEX      = "video_ai.silver.video_chunk_index"
LLM_ENDPOINT  = "databricks-meta-llama-3-3-70b-instruct"   # or your chosen FM API endpoint
TOP_K         = 4

w      = WorkspaceClient()
client = get_deploy_client("databricks")

# Comment: This function retrieves the top matching chunks from the vector search index.
def retrieve(question: str, k: int = TOP_K):
    res = w.vector_search_indexes.query_index(
        index_name=VS_INDEX,
        query_text=question,
        columns=["chunk_id", "video_id", "transcript",
                 "start_time", "end_time", "caption", "topic"],
        num_results=k,
    )
    cols = [c.name for c in res.manifest.columns]
    return [dict(zip(cols, row)) for row in res.result.data_array]

# Comment: Formats a timestamp into a readable HH:MM:SS value for citations.
def fmt_timestamp(seconds) -> str:
    s = int(float(seconds))
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}:{s % 60:02d}"

# Comment: Builds a compact context string from the retrieved chunks so the model has evidence to answer.
def build_context(chunks):
    blocks = []
    for i, c in enumerate(chunks, 1):
        ts = f"{fmt_timestamp(c['start_time'])}–{fmt_timestamp(c['end_time'])}"
        blocks.append(
            f"[{i}] video={c['video_id']} time={ts} topic={c.get('topic')}\n"
            f"{c['caption']}"
        )
    return "\n\n".join(blocks)

SYSTEM_PROMPT = """You are a video knowledge assistant.
Answer ONLY from the provided context. If the context does not contain the answer, say so.
Always end with a citation of the form: (video <video_id>, <start>–<end>).
Cite the single most relevant chunk."""

# Comment: This function orchestrates retrieval + generation for a single user question.
def answer(question: str):
    chunks  = retrieve(question)
    context = build_context(chunks)
    resp = client.predict(
        endpoint=LLM_ENDPOINT,
        inputs={
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",
                 "content": f"Question: {question}\n\nContext:\n{context}"},
            ],
            "temperature": 0.1,
            "max_tokens": 512,
        },
    )
    text = resp["choices"][0]["message"]["content"]
    return {"answer": text, "sources": chunks}

out = answer("What is this course about?")
print(out["answer"])

In [ ]:
def citation(chunk):
    start = int(float(chunk["start_time"]))
    # If videos are served over HTTP, many players accept #t=<seconds>
    return {
        "video_id":  chunk["video_id"],
        "timestamp": f"{fmt_timestamp(chunk['start_time'])}–{fmt_timestamp(chunk['end_time'])}",
        "deep_link": f"{chunk['video_path']}#t={start}",
    }

for c in out["sources"][:1]:
    print(citation(c))

In [ ]:
import mlflow
from mlflow.models.resources import (
    DatabricksVectorSearchIndex, DatabricksServingEndpoint,
)

with mlflow.start_run():
    logged = mlflow.pyfunc.log_model(
        name="video_rag_agent",
        python_model="/Workspace/Users/databricks_certifications@outlook.com/rag_agent.py",
        pip_requirements=["mlflow", "databricks-vectorsearch"],
        resources=[
            DatabricksVectorSearchIndex(index_name="video_ai.silver.video_chunk_index"),
            DatabricksServingEndpoint(endpoint_name="databricks-claude-3-7-sonnet"),
            DatabricksServingEndpoint(endpoint_name="databricks-gte-large-en"),
        ],
    )

In [ ]:
import pandas as pd, mlflow

eval_df = pd.DataFrame([
    {"request": "How does Delta Lake track table state?",
     "expected_facts": ["transaction log", "commits", "current table state"]},
    {"request": "What is discussed about streaming?",
     "expected_facts": ["streaming ingestion", "Auto Loader"]},
])

results = mlflow.evaluate(
    model=f"runs:/{logged.run_id}/video_rag_agent",
    data=eval_df,
    model_type="databricks-agent",
)
print(results.metrics)

In [ ]:
import mlflow
from databricks import agents

mlflow.set_registry_uri("databricks-uc")

uc_name = "video_ai.ai.video_rag_agent"
mv = mlflow.register_model(
    model_uri=f"runs:/{logged.run_id}/video_rag_agent",
    name=uc_name,
)

# Deploys a serving endpoint + a review app for stakeholder testing
agents.deploy(uc_name, mv.version)

In [ ]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady
from mlflow.deployments import get_deploy_client

endpoint_name = "agents_video_ai-ai-video_rag_agent"  # name shown by agents.deploy

w = WorkspaceClient()
for _ in range(60):
    ep = w.serving_endpoints.get(endpoint_name)
    if (ep.state.ready == EndpointStateReady.READY and
        ep.state.config_update == EndpointStateConfigUpdate.NOT_UPDATING):
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Endpoint {endpoint_name} not ready after 15 min")

client = get_deploy_client("databricks")
client.predict(
    endpoint=endpoint_name,
    inputs={"messages": [{"role": "user",
        "content": "Where does the instructor explain the purpsoe of the course?"}]},
)